<a href="https://colab.research.google.com/github/Bik4ram/semi_ai/blob/main/Semiconductor_AI_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Semiconductor AI Pilot — CPT → SFT → DPO → GRPO
**Run this on Google Colab: Runtime → Change runtime type → T4 GPU (free tier).**

This notebook runs the entire fixed pipeline end-to-end: real corpus → real data pipeline →
real Qwen2.5-Coder base model → CPT → SFT → DPO → GRPO (RL with a verilator-verified reward) →
export/serve. Every stage here actually executes — nothing is simulated or faked.

Run cells top to bottom. Each stage's model is saved to `models/<stage>-lora/` so you can stop
and resume, or skip stages you don't need.

## 0. Environment setup

In [1]:
%cd /content/semiconductor-ai-v2


# Clone again
!git clone https://github.com/Bik4ram/semi_ai

# Enter the repository
%cd semi_ai

# Verify
!pwd
!ls


[Errno 2] No such file or directory: '/content/semiconductor-ai-v2'
/content
Cloning into 'semi_ai'...
remote: Enumerating objects: 104, done.
remote: Counting objects: 100% (104/104), done.
remote: Compressing objects: 100% (97/97), done.
remote: Total 104 (delta 52), reused 34 (delta 4), pack-reused 0 (from 0)
Receiving objects: 100% (104/104), 1.10 MiB | 15.06 MiB/s, done.
Resolving deltas: 100% (52/52), done.
/content/semi_ai
/content/semi_ai
data	    models	      scripts
evaluation  README.md	      Semiconductor_AI_Pipeline.ipynb
inference   requirements.txt  training


In [ ]:
!nvidia-smi
!pip install -q unsloth trl peft bitsandbytes accelerate datasets huggingface_hub pyslang datasketch anthropic
!apt-get -y -qq install verilator > /dev/null
!verilator --version

Mon Jul 13 15:00:38 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   63C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
import os
PROJECT = "/content/semiconductor-ai-v2"
os.makedirs(PROJECT, exist_ok=True)
os.chdir(PROJECT)
# If you're running this notebook standalone (not from the delivered zip), fetch the scripts:
# !git clone <your-fork-of-this-repo> .   # or unzip an uploaded copy of the project files here
print("Working dir:", os.getcwd())

## 1. Collect the corpus
Clones picorv32, ibex, uvm-core, an OpenTitan peripheral subset, and riscv-arch-test —
all small, public, permissively-licensed.

In [ ]:
!bash scripts/01_clone_corpus.sh

## 2–4. Parse → dedupe → label

In [ ]:
!python3 scripts/02_parse_corpus.py
!python3 scripts/03_dedupe_chunk.py
!python3 scripts/04_label_chunks.py

## 5. Build the CPT corpus (raw text, for continued pretraining)

In [ ]:
!python3 scripts/05_build_cpt_corpus.py

## 6. Generate instruction pairs
Set `ANTHROPIC_API_KEY` below if you have one (optional — mechanical bug-injection pairs
work fully without it). Every SV/assertion output is verified with a real `verilator --lint-only`
run before being kept.

In [ ]:
!pip install -q openai
import os
from getpass import getpass

os.environ["GROQ_API_KEY"] = getpass("Enter your Groq API Key: ")

In [ ]:
from openai import OpenAI
import os

client = OpenAI(
    api_key=os.environ["GROQ_API_KEY"],
    base_url="https://api.groq.com/openai/v1"
)

response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {"role": "user", "content": "introduce yourself"}
    ],
)

print(response.choices[0].message.content)

In [ ]:
!git pull


In [ ]:
import os
# os.environ["ANTHROPIC_API_KEY"] = "sk-ant-..."   # uncomment and fill in to enable LLM augmentation
!python3 scripts/06_generate_pairs.py

## 7–8. Split train/val/test, build DPO preference pairs

In [ ]:
!python3 scripts/07_split_data.py
!python3 scripts/08_generate_dpo_pairs.py

## Environment check before training (do not skip)

In [ ]:
!python3 training/00_env_check.py

## STAGE 1/4 — Continued Pretraining (CPT)
Raw next-token prediction on your domain corpus. Absorbs vocabulary/style, not behavior.
Skip this cell if your corpus is small and you want to save time — SFT alone still works fine
on top of the plain base model.

In [ ]:
!git pull
!pwd

In [ ]:
!python3 training/10_cpt_train.py

## STAGE 2/4 — Supervised Fine-Tuning (SFT)
Set `START_FROM_CPT = True` inside `training/20_sft_train.py` first if you ran the CPT cell above
and want to chain from it. Default starts from the plain base model.

In [ ]:
!python3 training/20_sft_train.py

## STAGE 3/4 — DPO (preference optimization)

In [ ]:
!python3 training/30_dpo_train.py

## STAGE 4/4 — GRPO (RL with verilator-verified reward)
The slowest stage — samples multiple completions per prompt every step. Keep expectations
modest at this corpus size; the point is seeing the mechanics work end-to-end.

In [ ]:
!python3 training/40_grpo_train.py

## Evaluate every stage against the same frozen benchmark

In [ ]:
!python3 evaluation/evaluate.py --model Qwen/Qwen2.5-Coder-1.5B-Instruct --tag baseline
!python3 evaluation/evaluate.py --model models/sft-lora --tag sft
!python3 evaluation/evaluate.py --model models/dpo-lora --tag dpo
!python3 evaluation/evaluate.py --model models/grpo-lora --tag grpo

In [ ]:
import json
for tag in ["baseline","sft","dpo","grpo"]:
    path = f"evaluation/results/{tag}.json"
    if os.path.exists(path):
        data = json.load(open(path))
        avg_rouge = sum(r.get("rouge_l",0) for r in data)/max(1,len(data))
        print(f"{tag:10s} avg_rouge_l={avg_rouge:.3f}")

## Export, quantize (GGUF), and serve
Pass `dpo-lora` or `grpo-lora` depending on which stage you consider your final model.

In [ ]:
!bash inference/export_and_serve.sh dpo-lora

## Download your fine-tuned model
`models/merged-gguf/model.Q4_K_M.gguf` is your final, portable, quantized model — small enough
to download and run locally with llama.cpp or LM Studio.

In [ ]:
from google.colab import files
files.download("models/merged-gguf/model.Q4_K_M.gguf")